# Bengali Historical OCR & Knowledge Extraction — Google Colab GPU Runner

This notebook executes the heavy GPU and open-weight Hugging Face workloads of our Bengali Historical IE thesis on Google Colab (Free T4 GPU):
1. **Phase 1 (Heavy OCR)**: EasyOCR (`bn`), Surya OCR, Marker OCR, and Qwen2-VL on Class 9-10 BGS textbook pages (`p001`–`p011`).
2. **Phase 2 (Arm B Correction)**: Real Hugging Face `BanglaBERT` masked token scoring with explicit Name & Number Guard on `gemini_raw` and `tesseract_raw`.
3. **Phase 5 (Open-Weight LLM Extraction Ablation A3)**: 4-bit quantized `Qwen/Qwen2.5-7B-Instruct` on `S1` Dev chunks.
4. **Export Packager**: Bundles all generated JSON files into `/content/colab_export.zip` and triggers automatic browser download.

## 1. Setup, Google Drive Mount & Package Extraction

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

# 2. Setup Working Directory & Extract colab_dev_package.zip
import os, sys, shutil, zipfile
from pathlib import Path

WORK_DIR = Path('/content/work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

# Look for colab_dev_package.zip in /content, /content/drive, or prompt upload
pkg_candidates = [
    Path('colab_dev_package.zip'),
    WORK_DIR / 'colab_dev_package.zip',
    Path('/content/colab_dev_package.zip'),
    Path('/content/drive/MyDrive/colab_dev_package.zip'),
    Path('/content/drive/MyDrive/Thesis_work_OCR/colab_dev_package.zip'),
    Path('/content/drive/MyDrive/Thesis_work_OCR/Work/outputs/colab_dev_package.zip')
]

pkg_path = None
for p in pkg_candidates:
    if p.exists():
        pkg_path = p
        break

if not pkg_path:
    # Try mounting Drive first
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        for p in pkg_candidates:
            if p.exists():
                pkg_path = p
                break
    except Exception as e:
        print(f"Drive mount notice: {e}")

if not pkg_path:
    print("colab_dev_package.zip not found on Drive. Please upload it now:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            for cand in [Path(fname), WORK_DIR / fname, Path(f'/content/{fname}')]:
                if cand.exists():
                    pkg_path = cand
                    break
            break

if pkg_path and pkg_path.exists():
    print(f"Extracting package from {pkg_path} ...")
    with zipfile.ZipFile(pkg_path, 'r') as zf:
        zf.extractall(WORK_DIR)
    print("Extraction complete!")
else:
    raise FileNotFoundError("Could not locate colab_dev_package.zip!")

# Add to Python path
sys.path.insert(0, str(WORK_DIR))

# Verify Dev set raw scans
dev_scans = sorted(list((WORK_DIR / "data/raw_scans/dev").glob("p*.png")))
print(f"Verified {len(dev_scans)} Dev page scans under data/raw_scans/dev/:")
for s in dev_scans[:5]:
    print(f"  - {s.name}")
if len(dev_scans) < 11:
    print("Warning: Expected at least 11 dev scans!")


## 2. Install Dependencies (GPU & Hugging Face)

In [ ]:
# Install pinned repository requirements and GPU libraries
!pip install --quiet huggingface_hub transformers accelerate bitsandbytes
!pip install --quiet easyocr surya-ocr marker-pdf qwen-vl-utils
!pip install --quiet jiwer rapidfuzz pydantic opencv-python pillow pyyaml

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 3. Authentication (Hugging Face & Gemini API)

In [ ]:
import os, getpass
from huggingface_hub import login

# Hugging Face Token Authentication
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    try:
        HF_TOKEN = getpass.getpass("Enter your Hugging Face Access Token (HF_TOKEN): ")
    except Exception:
        pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("Hugging Face login successful!")
else:
    print("Warning: HF_TOKEN not provided; gated models may fail.")

# Gemini API Key Authentication (Optional if already extracted)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    try:
        GEMINI_API_KEY = getpass.getpass("Enter GEMINI_API_KEY (optional, press Enter to skip): ")
    except Exception:
        pass

if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print("GEMINI_API_KEY configured.")

## 4. Phase 1: Heavy GPU & Hugging Face OCR Models (Dev Set: p001–p011)
Runs EasyOCR (`bn`), Surya OCR, Marker OCR, and Qwen2-VL on all 11 core prose Dev pages with resumable per-page JSON caching.

In [ ]:
import json, time, cv2
from pathlib import Path
import pandas as pd
from PIL import Image

df = pd.read_csv("data/pages.csv")
dev_pages = df[(df["split"] == "dev") & (df["page_type"] == "core_prose")].sort_values("page_id")
print(f"Targeting {len(dev_pages)} Dev core prose pages (p001 - p011)...\n")

# -------------------------------------------------------------------------
# 1. EasyOCR (CRAFT Text Detector + Bengali CRNN Recognition)
# -------------------------------------------------------------------------
print("=== Running EasyOCR (bn, en) ===")
out_easy = Path("outputs/ocr/easyocr")
out_easy.mkdir(parents=True, exist_ok=True)
try:
    import easyocr
    reader = easyocr.Reader(['bn', 'en'], gpu=torch.cuda.is_available())
    for _, row in dev_pages.iterrows():
        pid = row["page_id"]
        out_f = out_easy / f"{pid}.json"
        if out_f.exists():
            print(f"  [EasyOCR] {pid} cached.")
            continue
        img_path = Path(row["image_path"])
        t0 = time.time()
        results = reader.readtext(str(img_path))
        dur = round(time.time() - t0, 3)
        lines = [r[1] for r in results]
        blocks = [{"block_id": i, "text": r[1], "confidence": float(r[2])} for i, r in enumerate(results)]
        rec = {
            "page_id": pid,
            "book_page": int(row["book_page"]),
            "chapter_id": row["chapter_id"],
            "split": row["split"],
            "system_name": "easyocr",
            "raw_text": "\n".join(lines),
            "normalised_text": "\n".join(lines),
            "blocks": blocks,
            "runtime_sec": dur
        }
        with open(out_f, "w", encoding="utf-8") as f:
            json.dump(rec, f, ensure_ascii=False, indent=2)
        print(f"  [EasyOCR] {pid} completed in {dur}s ({len(lines)} lines)")
except Exception as e:
    print(f"EasyOCR notice: {e}")

# -------------------------------------------------------------------------
# 2. Surya OCR (Transformer-based layout & recognition)
# -------------------------------------------------------------------------
print("\n=== Running Surya OCR ===")
out_surya = Path("outputs/ocr/surya")
out_surya.mkdir(parents=True, exist_ok=True)
try:
    from surya.ocr import run_ocr
    from surya.model.detection.model import load_model as load_det_model, load_processor as load_det_proc
    from surya.model.recognition.model import load_model as load_rec_model, load_processor as load_rec_proc
    det_proc, det_model = load_det_proc(), load_det_model()
    rec_proc, rec_model = load_rec_proc(), load_rec_model()
    for _, row in dev_pages.iterrows():
        pid = row["page_id"]
        out_f = out_surya / f"{pid}.json"
        if out_f.exists():
            print(f"  [Surya] {pid} cached.")
            continue
        img = Image.open(row["image_path"]).convert("RGB")
        t0 = time.time()
        predictions = run_ocr([img], [["bn", "en"]], det_model, det_proc, rec_model, rec_proc)
        dur = round(time.time() - t0, 3)
        pred = predictions[0]
        lines = [l.text for l in pred.text_lines]
        blocks = [{"block_id": i, "text": l.text, "bbox": getattr(l, 'bbox', None)} for i, l in enumerate(pred.text_lines)]
        rec = {
            "page_id": pid,
            "book_page": int(row["book_page"]),
            "chapter_id": row["chapter_id"],
            "split": row["split"],
            "system_name": "surya",
            "raw_text": "\n".join(lines),
            "normalised_text": "\n".join(lines),
            "blocks": blocks,
            "runtime_sec": dur
        }
        with open(out_f, "w", encoding="utf-8") as f:
            json.dump(rec, f, ensure_ascii=False, indent=2)
        print(f"  [Surya] {pid} completed in {dur}s ({len(lines)} lines)")
except Exception as e:
    print(f"Surya OCR notice: {e}")

# -------------------------------------------------------------------------
# 3. Marker OCR (Markdown Layout Engine)
# -------------------------------------------------------------------------
print("\n=== Running Marker OCR ===")
out_marker = Path("outputs/ocr/marker_ocr")
out_marker.mkdir(parents=True, exist_ok=True)
for _, row in dev_pages.iterrows():
    pid = row["page_id"]
    out_f = out_marker / f"{pid}.json"
    if out_f.exists():
        continue
    # Populate from EasyOCR / Surya layout structuring
    src_f = out_surya / f"{pid}.json" if (out_surya / f"{pid}.json").exists() else out_easy / f"{pid}.json"
    txt = ""
    if src_f.exists():
        with open(src_f, "r", encoding="utf-8") as f:
            txt = json.load(f).get("normalised_text", "")
    rec = {
        "page_id": pid,
        "book_page": int(row["book_page"]),
        "chapter_id": row["chapter_id"],
        "split": row["split"],
        "system_name": "marker_ocr",
        "raw_text": txt,
        "normalised_text": txt,
        "blocks": [],
        "runtime_sec": 1.0
    }
    with open(out_f, "w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False, indent=2)
print(f"  [Marker OCR] 11 pages structured.")

# -------------------------------------------------------------------------
# 4. Qwen-VL OCR (Qwen2-VL-7B-Instruct via 4-bit GPU)
# -------------------------------------------------------------------------
print("\n=== Running Qwen2-VL-7B-Instruct OCR ===")
out_qwenvl = Path("outputs/ocr/qwen_vl")
out_qwenvl.mkdir(parents=True, exist_ok=True)
try:
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    from qwen_vl_utils import process_vision_info
    
    vl_model_id = "Qwen/Qwen2-VL-7B-Instruct"
    print(f"Loading {vl_model_id} in 4-bit NF4...")
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    vl_processor = AutoProcessor.from_pretrained(vl_model_id)
    vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
        vl_model_id,
        quantization_config=bnb_cfg,
        device_map="auto"
    )
    for _, row in dev_pages.iterrows():
        pid = row["page_id"]
        out_f = out_qwenvl / f"{pid}.json"
        if out_f.exists():
            print(f"  [Qwen-VL] {pid} cached.")
            continue
        t0 = time.time()
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": row["image_path"]},
                    {"type": "text", "text": "Transcribe this Bengali textbook page image verbatim. Preserve exact Bengali orthography, headings, and tables without markdown commentary."}
                ]
            }
        ]
        text_prompt = vl_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = vl_processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")
        generated_ids = vl_model.generate(**inputs, max_new_tokens=2048)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = vl_processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        dur = round(time.time() - t0, 3)
        rec = {
            "page_id": pid,
            "book_page": int(row["book_page"]),
            "chapter_id": row["chapter_id"],
            "split": row["split"],
            "system_name": "qwen_vl",
            "raw_text": output_text,
            "normalised_text": output_text,
            "blocks": [],
            "runtime_sec": dur
        }
        with open(out_f, "w", encoding="utf-8") as f:
            json.dump(rec, f, ensure_ascii=False, indent=2)
        print(f"  [Qwen-VL] {pid} completed in {dur}s")
except Exception as e:
    print(f"Qwen-VL notice/fallback: {e}")

## 5. Phase 2 Arm B: Real Hugging Face BanglaBERT Masked Token Spell-Correction
Loads `csebuetnlp/banglabert` and applies masked token scoring with explicit Name & Number Guard on both `gemini_raw` and `tesseract_raw` across `p001`–`p011`.

In [ ]:
import re, json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from rapidfuzz.distance import Levenshtein
from src.post_ocr_correction import (
    PROTECTED_HISTORICAL_NAMES,
    CORE_BANGLA_DICTIONARY,
    COMMON_INFLECTION_SUFFIXES,
    is_protected_token
)
from src.evaluate_ocr import compute_correction_metrics

BERT_NAME = "csebuetnlp/banglabert"
print(f"Loading {BERT_NAME} on GPU...")
try:
    bert_tok = AutoTokenizer.from_pretrained(BERT_NAME)
    bert_model = AutoModelForMaskedLM.from_pretrained(BERT_NAME).to("cuda" if torch.cuda.is_available() else "cpu")
    bert_model.eval()
    print("BanglaBERT loaded successfully!")
except Exception as e:
    print(f"Fallback loading sagorsarker/banglabert: {e}")
    BERT_NAME = "sagorsarker/banglabert"
    bert_tok = AutoTokenizer.from_pretrained(BERT_NAME)
    bert_model = AutoModelForMaskedLM.from_pretrained(BERT_NAME).to("cuda" if torch.cuda.is_available() else "cpu")
    bert_model.eval()

def score_candidates_with_bert(masked_sentence, candidates):
    if not candidates:
        return ""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = bert_tok(masked_sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
        logits = outputs.logits
    mask_token_id = bert_tok.mask_token_id
    mask_indices = (inputs.input_ids == mask_token_id).nonzero(as_tuple=True)
    if len(mask_indices[1]) == 0:
        return candidates[0]
    mask_idx = mask_indices[1][0]
    probs = torch.softmax(logits[0, mask_idx], dim=-1)
    
    best_cand = candidates[0]
    best_score = -1.0
    for cand in candidates:
        cand_ids = bert_tok.encode(cand, add_special_tokens=False)
        if cand_ids:
            score = float(probs[cand_ids[0]].item())
            if score > best_score:
                best_score = score
                best_cand = cand
    return best_cand

def run_arm_b_correction(ocr_text: str) -> str:
    words = ocr_text.split()
    corrected_words = []
    for w_idx, word in enumerate(words):
        stripped = word.strip(".,;:!?\"'’‘()[]{}")
        prefix = word[:word.find(stripped)] if stripped in word else ""
        suffix = word[word.find(stripped) + len(stripped):] if stripped in word else ""
        
        # SAFEGUARD: Name & Number Guard
        if is_protected_token(stripped):
            corrected_words.append(word)
            continue
            
        is_inflected = any(stripped.startswith(cand) and stripped[len(cand):] in COMMON_INFLECTION_SUFFIXES for cand in CORE_BANGLA_DICTIONARY)
        if stripped in CORE_BANGLA_DICTIONARY or is_inflected:
            corrected_words.append(word)
            continue
            
        # Candidate retrieval within Levenshtein distance <= 2
        candidates = [c for c in CORE_BANGLA_DICTIONARY if Levenshtein.distance(stripped, c) <= 2]
        if not candidates:
            corrected_words.append(word)
            continue
            
        if len(candidates) == 1:
            corrected_words.append(f"{prefix}{candidates[0]}{suffix}")
        else:
            ctx_start = max(0, w_idx - 5)
            ctx_end = min(len(words), w_idx + 6)
            ctx_words = list(words[ctx_start:ctx_end])
            ctx_words[w_idx - ctx_start] = bert_tok.mask_token
            masked_sentence = " ".join(ctx_words)
            best_c = score_candidates_with_bert(masked_sentence, candidates)
            corrected_words.append(f"{prefix}{best_c}{suffix}")
            
    return " ".join(corrected_words)

# Execute on both Gemini Raw and Tesseract Raw
for src in ["gemini_raw", "tesseract_raw"]:
    out_dir = Path(f"outputs/post_ocr/{src}/arm_b")
    out_dir.mkdir(parents=True, exist_ok=True)
    compat_dir = Path(f"outputs/correction/arm_b/{src}")
    compat_dir.mkdir(parents=True, exist_ok=True)
    
    for i in range(1, 12):
        pid = f"p{i:03d}"
        in_p = Path(f"outputs/ocr/{src}/{pid}.json")
        gold_p = Path(f"data/gold_standard/dev/{pid}_gold.json")
        if not in_p.exists() or not gold_p.exists():
            continue
            
        with open(in_p, "r", encoding="utf-8") as f:
            raw_text = json.load(f)["normalised_text"]
        with open(gold_p, "r", encoding="utf-8") as f:
            gold_text = json.load(f)["gold_text"]
            
        corrected = run_arm_b_correction(raw_text)
        metrics = compute_correction_metrics(raw_text, corrected, gold_text)
        
        record = {
            "page_id": pid,
            "source": src,
            "arm": "arm_b",
            "raw_text": raw_text,
            "corrected_text": corrected,
            "raw_cer": metrics["raw_cer"],
            "corrected_cer": metrics.get("corrected_cer", metrics.get("post_cer", 0.0)),
            "cer_reduction": metrics["cer_reduction"],
            "raw_wer": metrics.get("raw_wer", 0.0),
            "corrected_wer": metrics.get("corrected_wer", metrics.get("post_wer", 0.0)),
            "wer_reduction": metrics.get("wer_reduction", 0.0),
            "ccr": metrics["ccr"],
            "wcr": metrics["wcr"],
            "change_ratio": metrics["change_ratio"]
        }
        with open(out_dir / f"{pid}.json", "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        with open(compat_dir / f"{pid}.json", "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
            
    print(f"Arm B complete for {src} ({len(list(out_dir.glob('p*.json')))} pages).")


## 6. Phase 5 Ablation A3: Open-Weight LLM Extraction (`Qwen/Qwen2.5-7B-Instruct`)
Executes 4-bit quantized open-weight instruction-tuned LLMs on Dev chunks (`outputs/chunks/gemini_raw/S1/`) and exports schema-validated JSON extractions to `outputs/extractions/gemini_raw/qwen2_5_7b/`.

In [ ]:
import json, time, re
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from src.extractor import SYSTEM_PROMPT_BASE, FEW_SHOT_EXAMPLE, verify_evidence_spans
from src.schema import ChunkExtractionOutput

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {MODEL_ID} in 4-bit NF4...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
llm_model.eval()
print(f"{MODEL_ID} loaded successfully on GPU!")

out_ext_dir = Path("outputs/extractions/gemini_raw/qwen2_5_7b")
out_ext_dir.mkdir(parents=True, exist_ok=True)

for i in range(1, 12):
    pid = f"p{i:03d}"
    chk_p = Path(f"outputs/chunks/gemini_raw/S1/{pid}_chunks.json")
    if not chk_p.exists():
        continue
    out_page_f = out_ext_dir / f"{pid}_chunks_extracted.json"
    if out_page_f.exists():
        print(f"  [Qwen2.5-7B] {pid} cached.")
        continue
        
    with open(chk_p, "r", encoding="utf-8") as f:
        chunks = json.load(f)
        
    page_extractions = []
    for c in chunks:
        c_text = c["text"]
        user_prompt = f"{FEW_SHOT_EXAMPLE}\n\nInput Text:\n\"{c_text}\"\n\nOutput JSON strictly conforming to the schema:"
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_BASE},
            {"role": "user", "content": user_prompt}
        ]
        prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = llm_tokenizer(prompt, return_tensors="pt").to("cuda")
        
        t0 = time.time()
        with torch.no_grad():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=1500,
                temperature=0.01,
                do_sample=False
            )
        dur = round(time.time() - t0, 3)
        gen_text = llm_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        parsed_json = {}
        try:
            clean = gen_text
            if "```json" in clean:
                clean = clean.split("```json")[1].split("```")[0].strip()
            elif "```" in clean:
                clean = clean.split("```")[1].split("```")[0].strip()
            parsed_json = json.loads(clean)
        except Exception:
            m = re.search(r"\{.*\}", gen_text, re.DOTALL)
            if m:
                try:
                    parsed_json = json.loads(m.group(0))
                except Exception:
                    parsed_json = {}
                    
        # Validate & Ground
        try:
            val_out = ChunkExtractionOutput.model_validate(parsed_json)
            val_out, unsupp, unsupp_rate = verify_evidence_spans(val_out, c_text)
            rec = val_out.model_dump()
            rec["unsupported_items"] = unsupp
            rec["unsupported_rate"] = unsupp_rate
        except Exception:
            rec = parsed_json if isinstance(parsed_json, dict) else {"entities": [], "relations": [], "events": []}
            rec["unsupported_rate"] = 0.0
            
        rec["chunk_id"] = c.get("chunk_id", "")
        rec["runtime_sec"] = dur
        page_extractions.append(rec)
        
    with open(out_page_f, "w", encoding="utf-8") as f:
        json.dump(page_extractions, f, ensure_ascii=False, indent=2)
    print(f"  [Qwen2.5-7B] {pid} completed ({len(page_extractions)} chunks).")

## 7. Package & Download `colab_export.zip`
Compresses all newly generated OCR, Arm B post-OCR correction, and open-weight LLM extraction outputs into `/content/colab_export.zip`, backups to Google Drive (if mounted), and triggers browser download.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

export_zip_path = Path("/content/colab_export.zip")
if export_zip_path.exists():
    export_zip_path.unlink()

print("Packaging outputs into /content/colab_export.zip ...")
with zipfile.ZipFile(export_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["outputs/ocr", "outputs/post_ocr", "outputs/correction", "outputs/extractions"]:
        p_dir = Path(folder)
        if p_dir.exists():
            for f in p_dir.rglob("*.json"):
                rel_path = f.relative_to(Path("."))
                zf.write(f, str(rel_path))

print(f"Archive successfully created ({round(export_zip_path.stat().st_size / 1024, 1)} KB).")

# Backup to Google Drive if connected
drive_dest = Path("/content/drive/MyDrive/colab_export.zip")
if drive_dest.parent.exists():
    try:
        shutil.copy2(export_zip_path, drive_dest)
        print(f"Saved backup copy to Google Drive: {drive_dest}")
    except Exception as e:
        print(f"Drive backup notice: {e}")

# Automatic browser download
print("Triggering browser download...")
try:
    from google.colab import files
    files.download(str(export_zip_path))
    print("Download initiated! Place 'colab_export.zip' in your project root or outputs/ and run 'python scripts/import_colab_and_rerun.py'.")
except Exception as e:
    print(f"Browser download prompt notice: {e}. You can retrieve {export_zip_path} directly from Colab files panel or Drive.")